In [59]:
import os, sys, subprocess, importlib
import numpy as np

# --- Project setup (works on Colab and locally) ---
_PROJECT_ROOT = None
for _candidate in [os.getcwd(), "/content/Distance-Estimator-NN", "/home/adsarver/Distance-Estimator-NN"]:
    if os.path.isfile(os.path.join(_candidate, "data.py")):
        _PROJECT_ROOT = _candidate
        break
if _PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot find project root containing data.py")

os.chdir(_PROJECT_ROOT)
sys.path.insert(0, _PROJECT_ROOT)

# Download dataset if not present (e.g. on Colab)
DATA_DIR = os.path.join(_PROJECT_ROOT, "rgbd-scenes-v2", "imgs")
if not os.path.isdir(DATA_DIR):
    print("Dataset not found — downloading rgbd-scenes-v2 ...")
    _url = "http://rgbd-dataset.cs.washington.edu/dataset/rgbd-scenes-v2/rgbd-scenes-v2_imgs.zip"
    subprocess.check_call(["wget", "-q", _url, "-O", "rgbd-scenes-v2_imgs.zip"])
    subprocess.check_call(["unzip", "-q", "rgbd-scenes-v2_imgs.zip"])
    os.remove("rgbd-scenes-v2_imgs.zip")
    print("Download complete.")

# Force-reload local modules in case they were cached
for _mod in ["data", "model"]:
    if _mod in sys.modules:
        importlib.reload(sys.modules[_mod])

print(f"Working directory: {os.getcwd()}")

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from data import RGBDDataset, scene_collate_fn
from model import DistanceNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- Hyperparameters ---
IMG_SIZE = 640
HIDDEN_SIZE = 128
LSTM_LAYERS = 2
MEMORY_LENGTH = 60
MEMORY_STRIDE = 1
LR = 1e-4
EPOCHS = 10
TBTT_LEN = 20          # truncation window: backprop every N frames
VAL_SPLIT = 0.2        # fraction of scenes held out for validation

# --- Scene train/val split ---
from glob import glob
from collections import defaultdict

all_files = sorted(glob(os.path.join(DATA_DIR, "*/*-color.png")))
_scenes_map = defaultdict(list)
for f in all_files:
    _scenes_map[os.path.basename(os.path.dirname(f))].append(f)

all_scene_names = sorted(_scenes_map.keys())
np.random.seed(42)
np.random.shuffle(all_scene_names)

n_val = max(1, int(len(all_scene_names) * VAL_SPLIT))
val_scene_names = all_scene_names[:n_val]
train_scene_names = all_scene_names[n_val:]

# --- DataLoaders (one scene per batch) ---
train_ds = RGBDDataset(DATA_DIR, scene_names=train_scene_names)
val_ds   = RGBDDataset(DATA_DIR, scene_names=val_scene_names, random_seed=123)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  collate_fn=scene_collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, collate_fn=scene_collate_fn, num_workers=0)

print(f"Scenes — train: {len(train_ds)}, val: {len(val_ds)}")
print(f"  Train scenes: {train_scene_names}")
print(f"  Val scenes:   {val_scene_names}")

Working directory: /content/Distance-Estimator-NN


ImportError: cannot import name 'scene_collate_fn' from 'data' (/content/Distance-Estimator-NN/data.py)

In [ ]:
# --- Model ---
model = DistanceNN(
    hidden_size=HIDDEN_SIZE,
    lstm_num_layers=LSTM_LAYERS,
    memory_length=MEMORY_LENGTH,
    memory_stride=MEMORY_STRIDE,
    img_size=IMG_SIZE,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"Model params: {sum(p.numel() for p in model.parameters()):,}")

Model params: 840,014,841


In [ ]:
def run_epoch(loader, model, optimizer, device, is_train=True):
    """
    TBTT epoch using a DataLoader that yields one scene dict per iteration.
    Hidden state carries across frames within a scene and is detached
    at every TBTT_LEN boundary.
    """
    mode = "Train" if is_train else "Val"
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_frames = 0

    for si, scene in enumerate(loader):
        rgb_list   = scene["rgb"]          # list[T] of (3, H, W)
        depth_list = scene["depth"]        # list[T] of (1, H, W)
        rgb_crops  = scene["rgb_crops"]    # list[T] of (3, ch, cw)
        depth_crops = scene["depth_crops"] # list[T] of (1, ch, cw)
        bboxes     = scene["bboxes"]       # list[T] of (4,)
        crop_dims  = scene["crop_dims"]    # list[T] of (h, w)
        T = len(rgb_list)

        if T < 2:
            continue

        model.reset_lstm()
        chunk_loss = torch.tensor(0.0, device=device)
        chunk_frames = 0

        for t in range(T):
            crop_h, crop_w = crop_dims[t]
            if crop_h < 2 or crop_w < 2:
                continue

            img = F.interpolate(rgb_list[t].unsqueeze(0), size=(IMG_SIZE, IMG_SIZE),
                                mode="bilinear", align_corners=False).to(device)
            bbox_t = bboxes[t].unsqueeze(0).to(device)
            obj_img = rgb_crops[t].unsqueeze(0).to(device)
            gt_depth = depth_crops[t].squeeze(0).to(device)

            gt_max = gt_depth.max()
            gt_norm = gt_depth / gt_max if gt_max > 0 else gt_depth

            if is_train:
                pred = model(img, bbox_t, crop_h, crop_w, obj_img=obj_img).squeeze(0)
                chunk_loss = chunk_loss + F.mse_loss(pred, gt_norm)
            else:
                with torch.no_grad():
                    pred = model(img, bbox_t, crop_h, crop_w, obj_img=obj_img).squeeze(0)
                    chunk_loss = chunk_loss + F.mse_loss(pred, gt_norm)

            chunk_frames += 1
            total_frames += 1

            # --- TBTT boundary: backprop and detach ---
            if is_train and chunk_frames % TBTT_LEN == 0:
                avg_loss = chunk_loss / TBTT_LEN
                optimizer.zero_grad()
                avg_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

                total_loss += chunk_loss.item()
                chunk_loss = torch.tensor(0.0, device=device)

                # Detach hidden states to cut the graph while preserving memory
                model.ctx_head.hx = tuple(h.detach() for h in model.ctx_head.hx)
                model.shape_head.hx = tuple(h.detach() for h in model.shape_head.hx)
                model.obj_head.hx = tuple(h.detach() for h in model.obj_head.hx)
                model.ctx_head.buffer = model.ctx_head.buffer.detach()
                model.shape_head.buffer = model.shape_head.buffer.detach()
                model.obj_head.buffer = model.obj_head.buffer.detach()

        # Flush remaining partial chunk
        leftover = chunk_frames % TBTT_LEN
        if leftover > 0:
            if is_train and chunk_loss.requires_grad:
                avg_loss = chunk_loss / leftover
                optimizer.zero_grad()
                avg_loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            total_loss += chunk_loss.item()

        avg_so_far = total_loss / max(total_frames, 1)
        print(f"  {mode} [{si+1}/{len(loader)}] {scene['scene']}: "
              f"{T} frames | running avg loss {avg_so_far:.6f}", end="\r", flush=True)

    print()
    return total_loss / max(total_frames, 1)

In [ ]:
# --- Training Loop (TBTT, scene-level streaming via DataLoader) ---
best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'='*50}\nEpoch {epoch}/{EPOCHS}\n{'='*50}")
    train_loss = run_epoch(train_loader, model, optimizer, device, is_train=True)
    val_loss   = run_epoch(val_loader,   model, optimizer, device, is_train=False)

    tag = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pt")
        tag = " * saved"

    print(f"Epoch {epoch:3d}/{EPOCHS} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}{tag}")

Epoch 1/10
Training epoch on 9142 samples...


OutOfMemoryError: CUDA out of memory. Tried to allocate 4.24 GiB. GPU 0 has a total capacity of 14.56 GiB of which 1.86 GiB is free. Including non-PyTorch memory, this process has 12.71 GiB memory in use. Of the allocated memory 8.75 GiB is allocated by PyTorch, and 3.82 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)